# 1) Imports

In [26]:
import json
import sys
import os
import pandas as pd
import random
import torch
from torch.utils.data import DataLoader

from torch.utils.data import Dataset
from torch.nn.utils.rnn import pad_sequence

from collections import defaultdict
from typing import Union, List

from typing import Dict, Any, Optional, Tuple
from tqdm import tqdm
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score
import numpy as np
from sklearn.model_selection import KFold


# 2) Defining Classes and Functions

In [27]:
class SequenceDataset(Dataset):
    def __init__(self, user_dict):

      # sorting in order to make apdding more efficient
      self.user_ids = sorted(user_dict.keys(), key=lambda x: len(user_dict[x]))
      self.sequences = [user_dict[user_id] for user_id in self.user_ids]

    def __getitem__(self, indices):
      sequence = self.sequences[indices]
      skill_one_hot_vectors = torch.tensor(np.array([item[0] for item in sequence[:-1]]), dtype=torch.float32)
      if len(sequence[0]) == 2:
          # Convert the list comprehension to a numpy array first, then to a tensor
          additional_features = torch.tensor(np.array([item[1] for item in sequence[:-1]]), dtype=torch.float32)
          additional_features = additional_features.unsqueeze(-1)  # Add the extra dimension
      else:
          # Convert the list comprehension to a numpy array first, then to a tensor
          additional_features = torch.tensor(np.array([list(item[1]) + [item[2]] for item in sequence[:-1]]), dtype=torch.float32)

      # Optimize labels creation
      labels = torch.tensor(np.array([list(item[0]) + [item[-1]] for item in sequence[1:]]), dtype=torch.float32)

      return skill_one_hot_vectors, additional_features, labels

    def __len__(self):
        return len(self.sequences)


# Define collate function
def collate_batch(batch):
    # Unpacking batches into sequences and labels
    skill_one_hot_vectors, additional_features, labels = zip(*batch)  # This will now work as expected

    # Convert sequences to padded tensors
    skill_one_hot_vectors_padded = pad_sequence(skill_one_hot_vectors, batch_first=True, padding_value=0)  # Pad inputs to max length
    additional_features_padded = pad_sequence(additional_features, batch_first=True, padding_value=0)  # Pad targets to max length
    labels_padded = pad_sequence(labels, batch_first=True, padding_value=0)  # Pad targets to max length
    lengths = torch.tensor([len(label) for label in labels])

    return skill_one_hot_vectors_padded, additional_features_padded, labels_padded, lengths


In [28]:
class KTDataset():
    def __init__(self,
                 df_answers,
                 df_skill_names,
                 renumber_skill_ids=True,
                 prepare_BKT=False,
                 prepare_DKT=False,
                 additional_columns=None):

        self.df_answers = df_answers
        self.df_skill_names = df_skill_names
        self.num_skills = len(df_skill_names)
        self.BKT_datadict = None
        self.DKT_datadict = None
        self.additional_columns = additional_columns
        self.num_other = 1 + (len(additional_columns) if additional_columns is not None else 0)

        if renumber_skill_ids:
            # Map original skill IDs to a continuous range starting from 1
            unique_skill_ids = sorted(df_skill_names['skill_id'].astype(int).unique())
            self.skill_id_mapping = {original_id: new_id for new_id, original_id in enumerate(unique_skill_ids, start=1)}
            # Apply the mapping to create the new column
            df_skill_names['skill_ids_renumbered'] = df_skill_names['skill_id'].map(self.skill_id_mapping)
        else:
            self.skill_id_mapping = None

        if prepare_BKT:
            self.create_BKT_datadict()

        if prepare_DKT:
            self.create_DKT_datadict()


    def return_ordered_ids(self, ids: Union[int, str, List[int], List[str]]) -> Union[int, List[int]]:
        """
        Returns renumbered skill IDs based on the mapping.

        Args:
            ids (Union[int, str, List[int], List[str]]): Original skill ID(s) to be renumbered.

        Returns:
            Union[int, List[int]]: Renumbered skill ID(s).
        """

        if isinstance(ids, (int, str)):
            return self.skill_id_mapping[int(ids)]
        elif isinstance(ids, list):
            return [self.skill_id_mapping[int(id)] for id in ids]
        else:
            raise TypeError("IDs must be an int, str, or a list of int/str.")


    def return_original_ids(self, ids: Union[int, str, List[int], List[str]]) -> Union[int, List[int]]:
        """
        Returns original skill IDs based on the renumbered skill IDs.

        Args:
            ids (Union[int, str, List[int], List[str]]): Renumbered skill ID(s) to be mapped back to the original IDs.

        Returns:
            Union[int, str, List[int], List[str]]: Original skill ID(s).
        """

        if isinstance(ids, (int, str)):
            return self.original_id_mapping.get(int(ids), None)  # Convert to integer for lookup
        elif isinstance(ids, list):
            return [self.original_id_mapping.get(int(id), None) for id in ids]  # Convert each to integer for lookup
        else:
            raise TypeError("IDs must be an int, str, or a list of int/str.")


    def create_BKT_datadict(self):
        """
        Converts Assistments data into a Bayesian Knowledge Tracing (BKT) dictionary.

        Groups data by skill ID and user ID, creating a structure where each skill maps
        to lists of user answer sequences.
        """

        # Create a defaultdict to collect answer sequences per skill
        skill_dict = defaultdict(list)

        for _, row in self.df_skill_names.iterrows():
            skill_id = row['skill_id']
            skill_name = row['skill_name']

            df_answers_filtered = self.df_answers[self.df_answers[str(skill_id)] == 1]
            # Group by user_id and collect sequences of correct answers
            answer_list = df_answers_filtered.groupby('user_id')['correct'].apply(list)

            # Collect all answer sequences for the current skill
            skill_dict[f"{skill_id} ({skill_name})"] = answer_list.tolist()

        # Convert defaultdict to a regular dictionary and return
        self.BKT_datadict = dict(skill_dict)


    def create_DKT_datadict(self):
        """
        Create a DKT dataset with one-hot vectors and additional columns if specified.
        """
        add_col = self.additional_columns

        # Dictionary to hold the data for each user
        dict_skills = defaultdict(list)

        # Iterate over each user group
        for user_id, user_group in self.df_answers.groupby('user_id'):
            # Create a list of tuples for each user's answers
            answer_list = []
            for _, row in user_group.iterrows():
                one_hot_vector = row.iloc[-self.num_skills:].values  # Extract the one-hot vector as an array

                # Collect additional column values as a list
                additional_info = row[add_col].values if add_col else []

                is_correct = row['correct']  # Extract if the answer was correct

                # Append the tuple (one-hot vector, additional info, correct flag)
                if add_col:
                    answer_list.append((one_hot_vector, additional_info, is_correct))
                else:
                    answer_list.append((one_hot_vector, is_correct))

            # Store the answer list in the dictionary under the user ID
            dict_skills[user_id] = answer_list

        self.DKT_datadict = dict(dict_skills)


In [ ]:

class CustomEmbedding(nn.Module):
    def __init__(self, num_skills, embed_dim):
        super(CustomEmbedding, self).__init__()
        # Linear layer for embedding
        self.linear = nn.Linear(num_skills, embed_dim)
        # Tanh activation
        self.tanh = nn.Tanh()

    def forward(self, x):
        # Perform the linear transformation
        embedded = self.linear(x)

        # Calculate the number of 1s (sum of the input binary vector)
        num_ones = torch.sum(x, dim=1, keepdim=True)

        # Avoid division by zero
        num_ones = torch.max(num_ones, torch.ones_like(num_ones))

        # Divide the embedded vector by the number of 1s
        embedded /= num_ones

        # Apply the Tanh activation
        return self.tanh(embedded)

In [29]:
class DKT(nn.Module):
    def __init__(self, num_skills, num_other, embed_dim, hid_size, num_hid_layers, drop_prob):
        super(DKT, self).__init__()

        # Custom embedding layer
        self.embedding = CustomEmbedding(num_skills, embed_dim)

        # RNN layer
        self.rnn = nn.LSTM(embed_dim + num_other, hid_size, num_hid_layers, batch_first=True)

        # Dropout layer for regularization
        self.dropout = nn.Dropout(p=drop_prob)

        # Output layer mapping hidden states to probabilities
        self.out = nn.Linear(hid_size, num_skills)

        # Sigmoid for probability output
        self.sigmoid = nn.Sigmoid()

    def forward(self, skills, other, lengths):
        """
        Forward pass for the DKT model.

        Args:
            inputs (Tensor): Input sequence of shape (batch_size, seq_len, 2), where each entry is (problem_id, correct).
            lengths (Tensor): Lengths of sequences (batch_size).

        Returns:
            Tensor: Output probabilities of shape (batch_size, seq_len, num_items).
        """
        # Embed the input sequence
        embedded = self.embedding(skills)

        concatenated = torch.cat((embedded, other), dim=-1)  # (batch_size, seq_length, embed_dim+other_dim)

        # Pack the padded sequence for the RNN
        packed_embedded = pack_padded_sequence(concatenated, lengths, batch_first=True, enforce_sorted=False)

        # RNN processing
        packed_output, _ = self.rnn(packed_embedded)

        # Unpack the sequence
        output, _ = pad_packed_sequence(packed_output, batch_first=True)

        # Apply dropout
        output = self.dropout(output)

        # Output layer for probabilities
        logits = self.out(output)

        # Sigmoid to convert logits to probabilities
        probabilities = self.sigmoid(logits)

        # Mask padded positions
        mask = torch.arange(probabilities.size(1)).expand(len(lengths), probabilities.size(1)) < lengths.unsqueeze(1)
        mask = mask.unsqueeze(-1).expand_as(probabilities)  # Shape: (batch_size, seq_len, num_items)
        mask = mask.to(probabilities.device)
        masked_output = probabilities * mask.float()  # Zero out the padded positions

        return masked_output


In [30]:
def calculate_DKT_loss(predictions_all, answers_with_labels, lengths):
    """
    Calculate the binary cross-entropy loss for a sequence of predictions and labels,
    while ignoring padded positions based on the given lengths.

    Args:
        predictions_all (Tensor): The model's predicted values with shape (batch_size, seq_len, num_skills).
        answers_with_labels (Tensor): Ground truth tensor with shape (batch_size, seq_len, num_skills + 1),
                                      where the last dimension contains problem IDs and correctness labels.
                                      The last element of each entry indicates whether the response was correct (1) or not (0).
        lengths (Tensor): A tensor of shape (batch_size,) indicating the lengths of each sequence in the batch.

    Returns:
        loss (Tensor): The total binary cross-entropy loss for the batch, ignoring padded positions.
    """
    # Extract the result (predictions) and labels from the inputs
    result, labels = _transform_to_correct_predictions(predictions_all, answers_with_labels)

    # Create a mask based on the sequence lengths, where 1 represents a valid position and 0 represents padding
    batch_size, seq_len = result.size()  # Assuming result is (batch_size, seq_len)
    mask = torch.arange(seq_len).expand(batch_size, seq_len) < lengths.unsqueeze(1)
    mask = mask.float()  # Convert to float for later multiplication

    # Apply the mask to the result and labels to ignore padded positions
    masked_result = result * mask
    masked_labels = labels * mask

    # Compute the binary cross-entropy loss for each sequence element
    loss = F.binary_cross_entropy(masked_result, masked_labels, reduction='none')

    # Average the loss over the non-padded positions
    masked_loss = loss.sum() / mask.sum()  # Normalize by the number of valid (non-padded) positions

    return masked_loss


def calculate_auc(predictions_all, answers_with_labels, lengths):
    """
    Calculate the AUC (Area Under the Curve) for a sequence of predictions and labels, considering valid (non-padded) data.

    Args:
        predictions_all (Tensor): The model's predicted values with shape (batch_size, seq_len, num_skills).
        answers_with_labels (Tensor): Ground truth tensor with shape (batch_size, seq_len, num_skills + 1),
                                     where the last element indicates correctness.
        lengths (Tensor): Lengths of sequences for each batch, to ignore padded values.

    Returns:
        auc (float): AUC score for the batch.
    """
    # Extract problem_ids and labels from the inputs
    predictions, labels = _transform_to_correct_predictions(predictions_all, answers_with_labels)

    # Move tensors to CPU and convert to NumPy arrays for AUC calculation
    predictions = predictions.cpu().detach().numpy()  # Move to CPU first
    labels = labels.cpu().detach().numpy()  # Same for labels
    lengths = lengths.cpu().detach().numpy()

    # Create mask to ignore padded values based on sequence lengths
    mask = np.arange(predictions.shape[1])[None, :] < lengths[:, None]  # Shape: (batch_size, seq_len)

    # Apply the mask to filter valid predictions and labels without using np.nan
    valid_predictions = predictions[mask]
    valid_labels = labels[mask]

    # Calculate AUC if valid data is present
    if valid_predictions.size > 0 and valid_labels.size > 0:
        auc = roc_auc_score(valid_labels, valid_predictions)
    else:
        auc = float('nan')  # Return NaN if no valid data for AUC calculation

    return auc


# Assuming test_loader is your DataLoader for the test set
def evaluate_auc(model, test_loader, device):
    model.eval()  # Set the model to evaluation mode
    auc_scores = []

    with torch.no_grad():  # Disable gradient computation for evaluation
        for skill_sequences, other_sequences, answers, lengths in test_loader:
            # Move inputs and answers to the correct device (GPU/CPU)
            skill_sequences = skill_sequences.to(device)
            other_sequences = other_sequences.to(device)
            answers = answers.to(device)
            lengths = lengths.to('cpu')

            # Forward pass through the model
            predictions = model(skill_sequences, other_sequences, lengths)  # Shape (batch_size, seq_len, num_items)

            # Calculate AUC for the current batch
            auc = calculate_auc(predictions, answers, lengths)
            auc_scores.append(auc)

    # Calculate the average AUC over all batches
    average_auc = sum(auc_scores) / len(auc_scores)
    return average_auc


def _transform_to_correct_predictions(predictions_all, answers_with_labels):
    # Extract problem IDs (skills) and correctness labels from the inputs
    problem_ids = answers_with_labels[..., :-1]  # Shape: (batch_size, seq_len, num_skills)
    labels = answers_with_labels[..., -1]   # Shape: (batch_size, seq_len)

    # Element-wise multiplication of predictions and problem IDs to select relevant predictions
    product = predictions_all * problem_ids  # Shape: (batch_size, seq_len, num_skills)

    # Sum over the skill dimension to aggregate predictions for each sequence step
    result_sum = product.sum(dim=2)  # Shape: (batch_size, seq_len)

    # Count the number of relevant skills (ones) for each step to use as a normalization factor
    num_ones = problem_ids.sum(dim=2)  # Shape: (batch_size, seq_len)

    # Avoid division by zero by aclipping at 1
    normalization_factor = torch.clamp(num_ones, min=1)

    # Normalize the summed result by the number of ones (relevant skills)
    result = result_sum / normalization_factor  # Shape: (batch_size, seq_len)

    return result, labels


In [49]:
def process(model, loader, device, optim=None):
    """
    Process the data in the given loader for either training or evaluation.

    Args:
        model: The model to be used for predictions.
        loader: DataLoader providing batches of input data and labels.
        device: The device (CPU or GPU) to run the computation on.
        optim: Optimizer for training (if provided). If None, the function performs evaluation.

    Returns:
        total_loss (float): The sum of all batch losses.
        average_auc (float): The average AUC score across all batches.
    """
    # Set model to training or evaluation mode
    if optim is not None:
        model.train()
        desc = 'Training'
    else:
        model.eval()
        desc = 'Evaluation'

    total_loss = 0
    total_auc = 0
    total_samples = 0


    model = model.to(device)

    with torch.no_grad() if optim is None else torch.enable_grad():

      # Iterate through the DataLoader with tqdm for progress tracking
      for skill_sequences, other_sequences, labels, lengths in tqdm(loader,
                                                                    file=sys.stdout,
                                                                    unit=' batches',
                                                                    desc=desc):
          # Move sequences and labels to the appropriate device
          skill_sequences = skill_sequences.to(device)
          other_sequences = other_sequences.to(device)
          labels = labels.to(device)
          lengths = lengths.to('cpu')

          # Forward pass
          outputs = model(skill_sequences, other_sequences, lengths)
          loss = calculate_DKT_loss(outputs, labels, lengths)
          auc = calculate_auc(outputs, labels, lengths)

          if optim is not None:  # Only during training
              optim.zero_grad()  # Reset gradients
              loss.backward()  # Backpropagation
              optim.step()  # Update parameters
              torch.cuda.empty_cache()

          batch_size = skill_sequences.size(0)  # Get the number of samples in the current batch

          # Weight the loss by the batch size
          total_loss += loss.item() * batch_size  # Accumulate weighted loss
          total_auc += auc * batch_size  # Accumulate weighted AUC
          total_samples += batch_size

    return total_loss / total_samples, total_auc / total_samples


def train_dkt(
    model_params: Dict[str, Any],
    lr: float,
    num_epochs: int,
    device: torch.device,
    train_loader: DataLoader,
    val_loader: Optional[DataLoader] = None
) -> Tuple[torch.nn.Module, float]:
    """
    Trains a Deep Knowledge Tracing (DKT) model using the provided parameters.

    Args:
        model_params (dict): A dictionary of model parameters to initialize the DKT model.
        lr (float): Learning rate for the optimizer.
        num_epochs (int): Number of epochs to train the model.
        device (torch.device): The device (e.g., 'cuda' or 'cpu') to run the training on.
        train_loader (DataLoader): DataLoader for the training dataset.
        val_loader (Optional[DataLoader]): DataLoader for the validation dataset. If None,
                                           training data is used for evaluation.

    Returns:
        Tuple[torch.nn.Module, float]: The trained DKT model and the best validation AUC achieved.

    Notes:
        - If `val_loader` is not provided, the training data is used for evaluation, which may
          lead to overestimation of the performance.
    """
    # Initialize the model and move it to the specified device
    model = DKT(**model_params)

    # Initialize the optimizer with the model's parameters and specified learning rate
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    list_train_loss = []
    list_train_auc = []
    list_val_loss = []
    list_val_auc = []

    # Training loop for the specified number of epochs
    for epoch in range(1, num_epochs + 1):
        print(f"\nEpoch {epoch}\n")

        # Training phase: Update the model using the training data
        process(model, train_loader, device, optimizer)

        # Validation phase: Evaluate the model on the validation dataset (if provided)
        if val_loader is not None:
            train_loss, train_auc = process(model, train_loader, device)
            val_loss, val_auc = process(model, val_loader, device)
            list_val_loss.append(val_loss)
            list_val_auc.append(val_auc)
            print(f'For the {epoch}. epoch test AUC is {val_auc}, test loss is {val_loss}.')
        else:
            # Use training data for evaluation if no validation DataLoader is provided
            print("No validation loader provided. Only training data for evaluation.")
            train_loss, train_auc = process(model, train_loader, device)
        list_train_loss.append(train_loss)
        list_train_auc.append(train_auc)

        print(f'For the {epoch}. epoch train AUC is {train_auc}, train loss is {train_loss}.')

    # Return the trained model and the best validation AUC achieved
    return model, list_train_loss, list_train_auc, list_val_loss, list_val_auc


In [32]:
def k_fold_cv_dkt(
    num_folds,
    model_params: Dict[str, Any],
    lr: float,
    num_epochs: int,
    device: torch.device,
    train_dict: Dict[int, Any],
    batch_size: int = 100,
    num_workers: int = 2
) -> float:
    """
    Perform k-fold cross-validation for the Deep Knowledge Tracing (DKT) model.

    Args:
        num_folds (int): Number of folds for cross-validation.
        model_params (Dict[str, Any]): Parameters for initializing the DKT model.
        lr (float): Learning rate for the optimizer.
        num_epochs (int): Number of epochs for training in each fold.
        device (torch.device): The device (e.g., 'cuda' or 'cpu') to run the training on.
        train_dict (Dict[int, Any]): Dictionary mapping keys to data for training.
        batch_size (int, optional): Batch size for training and validation. Defaults to 100.
        num_workers (int, optional): Number of subprocesses to use for data loading. Defaults to 2.

    Returns:
        float: The average validation AUC across all folds.

    Notes:
        - Assumes that the `train_dict` keys can be split into train and validation sets.
        - Requires an `AnswerSet` dataset and the `train_dkt` training function.
    """
    kf = KFold(n_splits=num_folds, shuffle=True, random_state=42)

    train_keys = list(train_dict.keys())

    val_auc_list = []

    for fold, (fold_train_idx, fold_val_idx) in enumerate(kf.split(train_keys)):
        print(f"\nFold {fold+1}/{num_folds}")
        fold_train_keys = [train_keys[i] for i in fold_train_idx]
        fold_val_keys = [train_keys[i] for i in fold_val_idx]

        fold_train_dict = {key: train_dict[key] for key in fold_train_keys}
        fold_train_dict = dict(sorted(fold_train_dict.items(), key=lambda item: len(item[1])))

        fold_val_dict = {key: train_dict[key] for key in fold_val_keys}
        fold_val_dict = dict(sorted(fold_val_dict.items(), key=lambda item: len(item[1])))

        fold_train_dataset = SequenceDataset(fold_train_dict)
        fold_train_loader = DataLoader(fold_train_dataset, batch_size=batch_size, collate_fn=collate_batch, pin_memory=True, num_workers=num_workers)

        fold_val_dataset = SequenceDataset(fold_val_dict)
        fold_val_loader = DataLoader(fold_val_dataset, batch_size=batch_size, collate_fn=collate_batch, pin_memory=True, num_workers=num_workers)

        *_, list_val_auc = train_dkt(
            model_params, lr, num_epochs, device,
            fold_train_loader, fold_val_loader
            )

        val_auc = list_val_auc[-1]

        val_auc_list.append(val_auc)
        print(f'For the {fold+1}. fold AUC is {val_auc}.')

    val_auc_avg = sum(val_auc_list) / num_folds

    return val_auc_avg


# 3) Model Training and Evaluation

## 3.1) Data imports and transforms


In [55]:
df_answers = pd.read_csv('df_answers.csv')
df_skill_names = pd.read_csv('df_skill_names.csv')

additional_columns = None
#additional_columns = ['ease']
#additional_columns = ['ease', 'ms_first_response', 'bottom_hint']

kt_dataset = KTDataset(df_answers, df_skill_names, prepare_DKT=True, additional_columns=additional_columns)

user_dict = kt_dataset.DKT_datadict
NUM_SKILLS = kt_dataset.num_skills
NUM_OTHER = kt_dataset.num_other


## 3.2) Seting constants

In [56]:
train_ratio = 0.8  # 80% for training, 20% for testing
NUM_EPOCHS = 15
BATCH_SIZE = 100
NUM_FOLDS = 5  # Number of folds for cross-validation
HID_SIZE = 200

EMBED_DIM = 3

finetune = True


In [ ]:
if additional_columns is None:
    model_version = "basic"
elif len(additional_columns) == 1:
    model_version = "ease"
else:
    model_version = "full"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


## 3.3) Splitting data into train and test sets

In [57]:
# Get all keys and shuffle them
keys = list(user_dict.keys())
random.shuffle(keys)

# Split keys into train and test
split_index = int(len(keys) * train_ratio)
train_keys = keys[:split_index]
test_keys = keys[split_index:]

# Create train and test dictionaries
train_dict = {key: user_dict[key] for key in train_keys}
test_dict = {key: user_dict[key] for key in test_keys}


## 3.4) Hyperparameter-tuning

In [ ]:
if finetune:
  configs = [
    {
        "learning_rate": lr,
        "model_params": {
            "num_skills": NUM_SKILLS,
            "num_other": NUM_OTHER,
            "embed_dim": EMBED_DIM,
            "hid_size": HID_SIZE,
            "num_hid_layers": num_hid_layers,
            "drop_prob": drop_prob
        },
    }
    for lr in [1e-3, 1e-4, 1e-5]  # 3 reasonable options for learning rate
    for num_hid_layers in [1, 2, 3]  # Hidden layers 1 or 2
    for drop_prob in [0.3, 0.4, 0.5]  # Dropout rate 0.3, 0.4, 0.5
    ]

  best_val_auc_avg = 0  # Track the best validation AUC
  best_config = None  # Track the best configuration

  for idx, config in enumerate(configs, 1):
      log_message = f"\nEvaluating Config {idx}/{len(configs)}"
      print(log_message)  # Print to console

      lr = config['learning_rate']
      model_params = config['model_params']

      val_auc_avg = k_fold_cv_dkt(
          num_folds=NUM_FOLDS,
          model_params=model_params,
          lr=lr,
          num_epochs=NUM_EPOCHS,
          device=device,
          train_dict=train_dict,
          batch_size=BATCH_SIZE,
          )

      # Update the global best if needed
      if val_auc_avg > best_val_auc_avg:
          best_val_auc_avg = val_auc_avg
          best_config = config  # Save the best configuration
          best_message = f"\nNew best model found: Config {idx}. Validation AUC: {best_val_auc_avg:.4f}"
          print(best_message)  # Print to console

  # Final output
  final_message = f"\nBest test AUC: {best_val_auc_avg:.4f}"
  print(final_message)

  final_config_message = f"\nBest Configuration: {best_config}"
  print(final_config_message)

else:
  # Load the configuration from the JSON file
  with open('dkt_best_config.json', 'r') as f:
      best_config = json.load(f)



Evaluating Config 1/27

Fold 1/5

Epoch 1

Evaluation: 100%|██████████| 6/6 [00:01<00:00,  4.45 batches/s]
For the 1. epoch AUC is 0.673652392761294, loss is 0.44927384952704114.

Epoch 2

Evaluation: 100%|██████████| 6/6 [00:01<00:00,  5.73 batches/s]
For the 2. epoch AUC is 0.6885692458078502, loss is 0.4425662060578664.

Epoch 3

Evaluation: 100%|██████████| 6/6 [00:01<00:00,  5.80 batches/s]
For the 3. epoch AUC is 0.6951292525495479, loss is 0.4392680327097575.

Epoch 4

Evaluation: 100%|██████████| 6/6 [00:01<00:00,  4.46 batches/s]
For the 4. epoch AUC is 0.6920046690645205, loss is 0.4375188648700714.

Epoch 5

Evaluation: 100%|██████████| 6/6 [00:01<00:00,  5.57 batches/s]
For the 5. epoch AUC is 0.6961868169987063, loss is 0.43640587230523425.

Epoch 6

Evaluation: 100%|██████████| 6/6 [00:01<00:00,  5.69 batches/s]
For the 6. epoch AUC is 0.7000555229164407, loss is 0.4353921363751094.

Epoch 7

Evaluation: 100%|██████████| 6/6 [00:01<00:00,  4.62 batches/s]
For the 7. epoc

## 3.5) Model training

In [48]:
# Training on the whole train set
train_dict = dict(sorted(train_dict.items(), key=lambda item: len(item[1])))
train_dataset = SequenceDataset(train_dict)
train_loader = DataLoader(train_dataset, batch_size=100, collate_fn=collate_batch, pin_memory=False, num_workers=2)

test_dict = dict(sorted(test_dict.items(), key=lambda item: len(item[1])))
test_dataset = SequenceDataset(test_dict)
test_loader = DataLoader(test_dataset, batch_size=100, collate_fn=collate_batch, pin_memory=False, num_workers=2)

# Log training details
training_message = "Training on the whole train set"
print(training_message)  # Print to console

best_model, list_train_loss, list_train_auc, list_val_loss, list_val_auc = train_dkt(
    model_params=best_config['model_params'],
    lr=best_config['learning_rate'],
    num_epochs=NUM_EPOCHS,
    device=device,
    train_loader=train_loader
)


Training on the whole train set

Epoch 1

Training: 100%|██████████| 30/30 [00:07<00:00,  4.07 batches/s]
No validation loader provided. Using training data for evaluation.
Evaluation: 100%|██████████| 30/30 [00:04<00:00,  6.76 batches/s]
For the 1. epoch AUC is 0.6962938881815839, loss is 0.5624951263268788.

Epoch 2

Training: 100%|██████████| 30/30 [00:05<00:00,  5.45 batches/s]
No validation loader provided. Using training data for evaluation.
Evaluation: 100%|██████████| 30/30 [00:05<00:00,  5.38 batches/s]
For the 2. epoch AUC is 0.7101586919037519, loss is 0.5553750862677892.

Epoch 3

Training: 100%|██████████| 30/30 [00:04<00:00,  6.41 batches/s]
No validation loader provided. Using training data for evaluation.
Evaluation: 100%|██████████| 30/30 [00:05<00:00,  5.33 batches/s]
For the 3. epoch AUC is 0.7071610067040799, loss is 0.5532138794660568.

Epoch 4

Training: 100%|██████████| 30/30 [00:05<00:00,  5.95 batches/s]
No validation loader provided. Using training data for ev

## 3.6) Model evaluation

In [50]:
train_loss = list_train_loss[-1]
train_auc = list_train_auc[-1]
test_loss = list_val_loss[-1]
test_auc = list_val_auc[-1]

# Example results for a model
results = {
    'Dataset': ['Train', 'Test'],
    'AUC': [train_auc, test_auc],
    'Loss': [train_loss, test_loss]
}

# Convert to DataFrame
df = pd.DataFrame(results)

print(df)


Evaluation: 100%|██████████| 8/8 [00:01<00:00,  4.91 batches/s]
  Dataset       AUC      Loss
0   Train  0.713485  0.547942
1    Test  0.713036  0.444262


## 3.7) Saving the results

In [ ]:
# Saving the model
model_save_path = f'dkt_{model_version}_model.pth'
torch.save(best_model.state_dict(), model_save_path)
model_save_message = f"Model saved to {model_save_path}"

print(model_save_message)  # Print to console

# Save as CSV
df.to_csv(f'dkt_{model_version}_model_results.csv', index=False)
